### **1. What is RAG?**

Retrieval-Augmented Generation (RAG) combines:

1. Retriever → Finds relevant information from documents.
2. Generator (LLM) → Generates answers using retrieved information.

### **RAG Pipeline**

Documents
    ->
Chunking
    ->
Embeddings
    ->
Vector Database
    ->
Retriever
    ->
Relevant Context
    ->
LLM
    ->
Final Answer

### **2. Install Required Libraries**

In [ ]:
!pip install -q \
langchain \
langchain-community \
langchain-openai \
faiss-cpu \
sentence-transformers \
pypdf \
tiktoken

### **3. Import Libraries**

In [ ]:
import os

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.embeddings import HuggingFaceEmbeddings

from langchain.vectorstores import FAISS

from langchain.chat_models import ChatOpenAI

from langchain.chains import RetrievalQA

### **4. Load Documents**

PDF Example

In [ ]:
loader = PyPDFLoader("knowledge_base.pdf")

documents = loader.load()

print("Total Pages:", len(documents))

### **5. Text Chunking**

Why Chunking? -> LLMs cannot process huge documents efficiently. Chunking breaks documents into smaller pieces.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

### **6. Generate Embeddings**

What are Embeddings?

Numerical vector representations of text.

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

### **7. Create Vector Database**

In [ ]:
#using FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)
vectorstore.save_local("faiss_index")

### **8. Load Existing Vector Database**

In [ ]:
vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

### **9. Similarity Search**

In [ ]:
query = "What is Retrieval Augmented Generation?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for doc in results:
    print(doc.page_content[:500])

### **10. Create Retriever**

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

### **11. Connect OpenAI LLM**

In [ ]:
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

llm = ChatOpenAI(
    model_name="gpt-4",
    temperature=0
)

### **12. Build RAG Chain**

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

### **13. Ask Questions**

In [ ]:
query = "Summarize the document."

response = qa_chain.run(query)

print(response)

### **14. View Retrieved Context**

In [ ]:
docs = retriever.get_relevant_documents(
    "Explain chapter 2"
)

for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}")
    print(doc.page_content[:500])

### **15. Better Prompting**

In [ ]:
from langchain.prompts import PromptTemplate

template = """
You are a helpful assistant.

Use ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context","question"],
    template=template
)

### **16. Advanced Retriever (MMR)**

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)

### **18. Conversational RAG**

In [ ]:
from langchain.chains import ConversationalRetrievalChain

chat_history = []

conversation = ConversationalRetrievalChain.from_llm(
    llm,
    retriever
)

response = conversation({
    "question":"What is RAG?",
    "chat_history":chat_history
})

print(response["answer"])

### **19. Source Citations**

In [ ]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

result = qa("Explain transformers")

print(result["result"])

for doc in result["source_documents"]:
    print(doc.metadata)

### **20. Hybrid Search (Dense + Keyword)**

In [ ]:
from rank_bm25 import BM25Okapi

texts = [doc.page_content for doc in chunks]

tokenized = [t.split() for t in texts]

bm25 = BM25Okapi(tokenized)

query = "What is deep learning"

scores = bm25.get_scores(query.split())

top_docs = sorted(
    zip(scores,texts),
    reverse=True
)[:5]

### **21. Reranking**

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

pairs = [
    [query, doc.page_content]
    for doc in results
]

scores = reranker.predict(pairs)

best_docs = sorted(
    zip(scores, results),
    reverse=True
)